In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/drive/MyDrive/binary_dataset.zip /content

In [ ]:
!unzip binary_dataset.zip

Streaming output truncated to the last 5000 lines.
  inflating: binary_dataset/NonPlant/cat_0514.jpg  
  inflating: binary_dataset/NonPlant/cat_0515.jpg  
  inflating: binary_dataset/NonPlant/cat_0516.jpg  
  inflating: binary_dataset/NonPlant/cat_0517.jpg  
  inflating: binary_dataset/NonPlant/cat_0518.jpg  
  inflating: binary_dataset/NonPlant/cat_0519.jpg  
  inflating: binary_dataset/NonPlant/cat_0520.jpg  
  inflating: binary_dataset/NonPlant/cat_0521.jpg  
  inflating: binary_dataset/NonPlant/cat_0522.jpg  
  inflating: binary_dataset/NonPlant/cat_0523.jpg  
  inflating: binary_dataset/NonPlant/cat_0524.jpg  
  inflating: binary_dataset/NonPlant/cat_0525.jpg  
  inflating: binary_dataset/NonPlant/cat_0526.jpg  
  inflating: binary_dataset/NonPlant/cat_0527.jpg  
  inflating: binary_dataset/NonPlant/cat_0528.jpg  
  inflating: binary_dataset/NonPlant/cat_0529.jpg  
  inflating: binary_dataset/NonPlant/cat_0530.jpg  
  inflating: binary_dataset/NonPlant/cat_0531.jpg  
  inflating: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import copy
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
data_dir = "binary_dataset"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

dataset = datasets.ImageFolder(data_dir, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size]
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Classes:", dataset.classes)

Classes: ['NonPlant', 'Plant']


In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

for param in model.features.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(model.last_channel, 2)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 160MB/s]


In [ ]:
epochs = 5
best_acc = 0.0
best_wts = copy.deepcopy(model.state_dict())

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    model.train()
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    print("Train Accuracy:", train_acc)

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    print("Validation Accuracy:", val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        best_wts = copy.deepcopy(model.state_dict())
        torch.save(best_wts, "binary_model.pth")
        print("Saved Best Binary Model")

print("Best Validation Accuracy:", best_acc)


Epoch 1/5
Train Accuracy: 98.40443981963233
Validation Accuracy: 99.93065187239945
Saved Best Binary Model

Epoch 2/5
Train Accuracy: 99.79188345473464
Validation Accuracy: 99.86130374479889

Epoch 3/5
Train Accuracy: 99.77454040929587
Validation Accuracy: 99.93065187239945

Epoch 4/5
Train Accuracy: 99.86125563648977
Validation Accuracy: 99.93065187239945

Epoch 5/5
Train Accuracy: 99.94797086368366
Validation Accuracy: 99.86130374479889
Best Validation Accuracy: 99.93065187239945


In [ ]:
with open("binary_classes.json", "w") as f:
    json.dump(dataset.classes, f)